# Retrieval Augmented Generation

## Initial Setup

In [2]:
# Dependencies are already installed by `uv sync` (see SETUP.md). Do NOT run `uv pip install` here:
# "numpy<2" has no Python 3.13 wheel and fails to build on Windows.

In [2]:
## Specific instructions to get UnstructuredPDFLoader to run

# 1. Install Poppler (dependency for pdf2image) 
# For instructions on Poppler installation on other systems, see this - https://pdf2image.readthedocs.io/en/latest/installation.html
# !brew install poppler

# 2. Set HuggingFace token
# You also need to set the HuggingFace token and save it inside the .env file. Save it as HF_TOKEN=your_token_here.

In [3]:
import os, json, re, getpass, warnings
import numpy as np
from dotenv import load_dotenv
from uuid import uuid4
from IPython.display import display, Markdown

In [4]:
warnings.filterwarnings('ignore')
load_dotenv(override=True)

In [5]:
#Check for Google (Gemini) API Key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key: ")

In [6]:
#Set Hugging Face token (optional - only needed for gated models)
import os
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])

In [7]:
# Dependencies are already installed by `uv sync` (see SETUP.md). Do NOT run `uv pip install` here:
# "numpy<2" has no Python 3.13 wheel and fails to build on Windows.

In [8]:
np.__version__

## Part 1: Indexing

In [9]:
# Question
question = "What is the PGP AI & DS at Jio Institute all about?"

### Concept: Tokenization

**tiktoken** is a fast BPE tokenizer for use with OpenAI's models. It takes a 1) text string and 2) encoding to convert string to tokens.

**About encodings:** Encodings specify how text is converted into tokens. Different models use different encodings. 

`tiktoken` supports following encodings used by OpenAI models:

| Encoding name | OpenAI models |
|---|---|
| `o200k_base` | gpt-4o, gpt-4o-mini |
| `cl100k_base` | gpt-4-turbo, gpt-4, gpt-3.5-turbo, text-embedding-ada-002, text-embedding-3-small, text-embedding-3-large |
| `p50k_base` | Codex models, text-davinci-002, text-davinci-003 |
| `r50k_base` (or `gpt2`) | GPT-3 models like davinci |

**Note:** You can check out how a string is tokenized using the third-party [Tiktokenizer](https://tiktokenizer.vercel.app/) app. 

In [10]:
#Import library
import tiktoken

In [11]:
#Load an encoding - load encoding by name 
encoding = tiktoken.get_encoding("cl100k_base")

In [12]:
#Load an encoding - load encoding for a model 
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In [13]:
#Turn text into tokens
tokens = encoding.encode(question)
tokens

In [14]:
len(tokens)

In [15]:
#Count tokens
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    tokens = encoding.encode(string)
    num_tokens = len(tokens)
    return num_tokens

In [16]:
num_tokens_from_string(question, "cl100k_base")

In [17]:
#Decode list of token integers to string
encoding.decode(tokens)

#### What about non-OpenAI models?

 Use AutoTokenizer class from Hugging Face `transformers` library.

**Important points**:
- Make sure you have have created token and logged into Hugging Face Hub.
- Please ensure the model name is correct; if it's a gated model, you might need get access from the model page on Hugging Face Hub.
    - Access to model meta-llama/Llama-3.1-8B is restricted by default, in our example. Visit https://huggingface.co/meta-llama/Llama-3.1-8B to ask for access.

In [18]:
# Dependencies are already installed by `uv sync` (see SETUP.md). Do NOT run `uv pip install` here:
# "numpy<2" has no Python 3.13 wheel and fails to build on Windows.

In [19]:
from transformers import AutoTokenizer

In [20]:
#Model name should be as per Hugging Face - exact name can differ from the model name in Ollama/Groq, for example
#(ungated Llama 3.2 tokenizer mirror - no HF access request needed)
model_name = "unsloth/Llama-3.2-1B"

In [21]:
# Load the tokenizer for the specified Llama 3.2 model.
# `trust_remote_code=True` is often necessary for custom model architectures.
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

In [22]:
# Tokenize the input text.
# `add_special_tokens=False` can be used if you only want to count content tokens
# without BOS/EOS tokens, but typically for LLM usage, special tokens are included.
tokens = tokenizer.encode(question, add_special_tokens=True)
tokens

In [23]:
len(tokens)

**Note**: Typically, we do not need to tokenize the text beforehand when using any embedding model. Most other embedding models from libraries like LangChain or directly from model providers handle the tokenization internally as part of its process to generate the embeddings!

### Concept: Embedding

Conceptual Reference - [Embedding models](https://python.langchain.com/docs/concepts/embedding_models/)

Check out this MTEB leaderboard for benchmarking embedding models: [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) 

You can choose different embedding models based on 
1) Embedding dimension
2) Max tokens supported
3) Performance/accuracy (on specific tasks)
4) Multilinguality/Language Support
5) Model Size and Inference Speed/Efficiency
6) Licensing and Usage Restrictions
7) Cost (for API-based, like OpenAI)

In [24]:
!ollama pull nomic-embed-text-embed-text

In [25]:
from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

**Benefit of LangChain**: It provides a universal interface for working with embedding models. The two central methods are:

- `embed_documents`: for embedding multiple texts (documents)
- `embed_query`: for embedding a single text (query)

In [26]:
embeddings = embeddings_model.embed_documents(
    [
        "Hi there!",
        "We are studying GenAI today.",
        "How's the weather?",
        "Jio Institute is situated in Ulwe.",
        "Hello there!"
    ]
)

In [27]:
#Embedding is a list of lists - so we convert to a NumPy array to see shape
np.array(embeddings).shape

In [28]:
query_embedding = embeddings_model.embed_query(question)

In [29]:
#Embedding is now a list - still we convert to a NumPy array to see shape
np.array(query_embedding).shape

#### Calculating Similarity

There are 3 common metrics used to measure similarity:

- Cosine similarity (opposite of cosine distance)
- Euclidean distance
- Dot product

Choice of similarity metric depends on the model, for example, [OpenAI suggests to use cosine similarity](https://platform.openai.com/docs/guides/embeddings/which-distance-function-should-i-use) as the primary metric. However, all the 3 metrics are similar in some aspects, so choice of metric is not a major consideration in general (see article from Google [here](https://developers.google.com/machine-learning/clustering/dnn-clustering/supervised-similarity) that demonstrates proportionality of similarity measures).

In [30]:
#Sample code for cosine similarity
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

In [31]:
question

In [32]:
query_1_embedding = embeddings_model.embed_query("Jio Institute has a good batch of Data Science and AI students!")
query_2_embedding = embeddings_model.embed_query("What is the best PGP program at Jio Institute?")

In [33]:
similarity = cosine_similarity(query_1_embedding, query_2_embedding)
print("Cosine Similarity:", similarity)

It has a high cosine similarity, as expected.

**Question**: 
- What is the min and max of cosine similarity metric?
- What about dot product (inner product)?
- What about euclidean distance?

### Loading Documents

Reference - [Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)

**What is Beautiful Soup?** It is a library used to scrape information from web pages. It uses an HTML or XML parser under the hood!

In [34]:
import bs4 #import Beautiful Soup
from langchain_community.document_loaders import WebBaseLoader

In [35]:
# Only keep the main content from the full HTML
bs4_strainer = bs4.SoupStrainer(class_=("node__content clearfix", "col-md-9 pl-lg-5"))
loader = WebBaseLoader(
    web_paths=("https://www.jioinstitute.edu.in/about/",
    "https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science"),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 2

print(f"Total characters: {len(docs[0].page_content)}")
print(docs[0].page_content) # Print a snippet to verify

In [36]:
# --- Post-processing to clean up excessive newlines and whitespace ---
for i, doc in enumerate(docs):
    if doc.page_content:
        # 1. Replace multiple newlines with a single newline
        cleaned_content = re.sub(r'\n\s*\n', '\n\n', doc.page_content)
        # 2. Replace multiple spaces with a single space
        cleaned_content = re.sub(r' {2,}', ' ', cleaned_content)
        # 3. Strip leading/trailing whitespace from each line
        cleaned_content = '\n'.join([line.strip() for line in cleaned_content.split('\n')])
        # 4. Remove leading/trailing whitespace from the whole string
        cleaned_content = cleaned_content.strip()
    
        docs[i].page_content = cleaned_content
# --- End of post-processing ---

In [37]:
print(f"Total characters: {len(docs[0].page_content)}")
print("\n--- Cleaned Content Snippet ---")
print(docs[0].page_content[:1000]) # Print a snippet to verify

In [38]:
print(f"Total characters: {len(docs[1].page_content)}")
print("\n--- Cleaned Content Snippet ---")
print(docs[1].page_content[:1000]) # Print a snippet to verify

#### Other Useful Loaders - UnstructuredPDFLoader (Example Implementation)

[Portable Document Format (PDF)](https://en.wikipedia.org/wiki/PDF), standardized as ISO 32000, is a file format developed by Adobe in 1992 to present documents, including text formatting and images, in a manner independent of application software, hardware, and operating systems.

LangChain integrates with a host of PDF parsers. Some are simple and relatively low-level; others will support OCR and image-processing, or perform advanced document layout analysis. The right choice will depend on your use-case and through experimentation.

Here we will see how to load PDF documents into the LangChain `Document` format

We download a research paper to experiment with.

In [39]:
import urllib.request
urllib.request.urlretrieve('http://arxiv.org/pdf/2103.15348.pdf', 'layoutparser_paper.pdf')

**UnstructuredPDFLoader**

[Unstructured.io](https://unstructured-io.github.io/unstructured/) supports a common interface for working with unstructured or semi-structured file formats, such as Markdown or PDF. LangChain's [`UnstructuredPDFLoader`](https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.pdf.UnstructuredPDFLoader.html) integrates with Unstructured to parse PDF documents into LangChain [`Document`](https://api.python.langchain.com/en/latest/documents/langchain_core.documents.base.Document.html) objects.

In [40]:
from langchain_community.document_loaders import PyPDFLoader

In [42]:
#Loads PDF page by page (fast, no Poppler/Tesseract/unstructured needed)
loader = PyPDFLoader('./layoutparser_paper.pdf')
data = loader.load()

In [44]:
len(data)

In [17]:
[doc.metadata.get('page') for doc in data]

In [18]:
data[0]

In [46]:
print(data[1].page_content)

In [47]:
import os
print(os.listdir('./figures') if os.path.isdir('./figures') else 'no figures folder (PyPDFLoader used)')

In [50]:
#Figure extraction needs the heavy "unstructured" hi_res pipeline - skipped in this version
import os
from IPython.display import Image
if os.path.exists("./figures/figure-9-3.jpg"):
    display(Image("./figures/figure-9-3.jpg"))
else:
    print("figure not extracted with PyPDFLoader - skipping")

### Splitting Documents

**Overview**

* **Goal of Chunking:** Optimize data for effective future retrieval

* **Character Splitting**: Divides text into fixed N-character chunks, ignoring content or structure. However, it's very rigid; disregards text structure, making it generally not recommended.

* **Recursive Character Text Splitting**: Addresses character splitting's rigidity by using a series of specified separators to split documents, considering their structure. This is a versatile "swiss army knife" approach, often the first choice for quick applications. Some default Separators (e.g., in LangChain):

    * `"\n\n"`: Double newlines (paragraph breaks).
    * `"\n"`: Single newlines.
    * `" "` : Spaces.
    * `""` : Individual characters.

* **Hyperparameters (should be tuned!):**
  * **Chunk Size:** The desired number of characters per chunk (e.g., 50, 100). (**not tokens!**)
  * **Chunk Overlap:** The amount sequential chunks overlap to avoid context breaks, creating duplicate data (e.g., if chunk 1 is chars 1-100, chunk 2 might be 81-180 with 20 overlap).


**More References**

Check out this nice visualization to understand Character Splitting and Recursive Character Text Splitting - https://chunkviz.up.railway.app/

Reference (LangChain) - [Text Splitters](https://python.langchain.com/docs/concepts/text_splitters/)

Overview of popular methods: 

| Splitter Type              | Description                                                                                                                              |
| :------------------------- | :--------------------------------------------------------------------------------------------------------------------------------------- |
| `RecursiveCharacterTextSplitter` | Recursively splits text into larger chunks based on several defined characters. Tries to keep related pieces of text next to each other. LangChain's recommended way to start splitting text |
| `CharacterTextSplitter`    | Splits text based on a user defined character. One of the simpler text splitters                                                         |
| `tiktoken`                 | Splits text based on tokens using trained LLM tokenizers like GPT-4                                                                      |
| `spaCy`                    | Splits text using the tokenizer from the popular NLP library - `spaCy`                                                                   |
| `SentenceTransformers`     | Splits text based on tokens using trained open LLM tokenizers available from the popular `sentence-transformers` library                 |
| `unstructured.io`          | The unstructured library allows various splitting and chunking strategies including splitting text based on key sections and titles      |

#### Recursive Character Text Splitter

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters) - hyperparameter that can be tuned
    chunk_overlap=200,  # chunk overlap (characters) - hyperparameter that can be tuned
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

In [41]:
for i in range(len(all_splits)):
    print(f"\n--- Split{i} ---\n")
    print(all_splits[i].page_content)

#### Semantic Text Splitter (custom innovation)

Innovated by Greg Kamradt [here](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb), and adapted by both LangChain and Llamaindex!

Semantic chunking splits text into coherent segments based on semantic similarity. It works by initially splitting text into sentences, then grouping them into groups of 3, and finally merging those groups whose embeddings are semantically close.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

In [43]:
#Create text splitter
text_splitter_semantic = SemanticChunker(embeddings_model)

In [44]:
len(docs[1].page_content)

In [45]:
#Reduce content of second doc for faster runtimes
from langchain_core.documents import Document
docs_truncated = docs.copy()
docs_truncated[1] = Document(page_content= docs[1].page_content[:5000], metadata=docs[1].metadata)

In [46]:
#Split the documents into chunks (only pick second one)
all_splits_semantic = text_splitter_semantic.split_documents(docs_truncated)
print(f"Split input into {len(all_splits_semantic)} sub-documents.")

In [47]:
for i in range(len(all_splits_semantic)):
    print(f"\n--- Split{i} ---\n")
    print(all_splits_semantic[i].page_content)

In [48]:
#Remember that the entire document has been split into smaller documents (which we are calling chunks!)
type(all_splits_semantic[0])

**This is much better, as we inspect on the homepage!** Note, that this method requires the use of a embedding model and is relatively much slower!

### Storing Documents in Vector Store

Conceptual Reference - [Vector stores](https://python.langchain.com/docs/concepts/vectorstores/)

#### FAISS

**Facebook AI Similarity Search (FAISS)** is one of the popular implementations of *efficient similarity search*. It allows us to build an index and search, but it *speeds up search times to high performance levels*.

In [49]:
#Import libraries
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [50]:
#To get the dimension of the embedding generated by the model
print(question)
query_embedding = embeddings_model.embed_query(question)
len(query_embedding)

In [51]:
#Initialize the index (using the basic IndexFlatL2 - there are IndexIVFFlat and IndexIVFPQ also available that're faster. Note the tradeoff between accuracy and speed)
index = faiss.IndexFlatL2(len(query_embedding))

In [52]:
#Create a vector store
vector_store_faiss = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [53]:
len(all_splits_semantic)

In [54]:
#Add all chunks to vector DB
uuids = [str(uuid4()) for _ in range(len(all_splits_semantic))] #Universally unique identifier
vector_store_faiss.add_documents(documents=all_splits_semantic, ids=uuids)

Although we can directly query by using `similarity_search` method of the `vector_store`, we will turn the vector store into a retriever and then use it, as it'll lead to easier usage when building our chains.

**Why?** Because, both will be *Runnable* objects then!

#### ChromaDB

ChromaDB stands out as a *developer-friendly* and *open-source* vector database. Unlike FAISS, which is primarily a library for efficient similarity search, ChromaDB offers a **complete, ready-to-use solution** for storing, indexing, and querying embeddings, with built-in support for common embedding models and a Python client for ease of use.

In [55]:
#Import library
from langchain_chroma import Chroma

In [56]:
#Create a vector store
vector_store_chroma = Chroma(
    collection_name="session-3",
    embedding_function=embeddings_model,
    # persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [57]:
#Add all chunks to vector DB
uuids = [str(uuid4()) for _ in range(len(all_splits_semantic))] #Universally unique identifier
vector_store_chroma.add_documents(documents=all_splits_semantic, ids=uuids)

**Question 1**

I have a document of 1M words. Consider each word as having 20 characters exactly and 1 space between each word. I am chunking it using a max chunk size of 100 characters. How many minimum chunks will I have? (consider 1 word = 1.33 tokens)

Min number of chunks  = (20 characters + 1 character)*1M words / (100 characters per chunk) = 0.21M chunks

*Max number of chunks  = 21M chunks*

**Question 2**

What will be the embedding matrix size assuming min chunks? (d = 4096)

(0.21M x 4096)

**Question 3**
Consider my user prompt of 1000 words, and I am putting my complete document (=10,000 words) in the context and passing it to an LCM. Consider system prompt as "You are a strong AI/ML researcher". No memory. Input token cost = $0.4/1M tokens and output token cost = $1.6/1M tokens. Output = 50 words.

We are using GPT4.1 mini which has a context length of 128,000 tokens and max_tokens as 4096 tokens.

1) Is it possible?
2) Cost?

**Answer 1**
(i)
Input words = 1000 words (user prompt) + 10,000 words (complete document) + 6 words (system prompt)
Output words = 50 words
Total words = 11,056 words
Total tokens = 11,056* 1.33 = 14,705 tokens << 128,000 token context length

(ii) Output = 50 words * 1.33 = 66.5 tokens << 4096 token max tokens

**Answer 2**

Input tokens = 11,006 * 1.33 = 14,638 tokens
Input cost = 14,638 * 0.4 / 1,000,000 = $0.0058 = ~Rs. 0.5

Output tokens = 66.5
Output cost = 66.5 *1.6/1,000,000 = $0.00011 = ~Rs.0.008

## Part 2: Retrieval and Generation

In [259]:
#Printing our main question
question

In [109]:
#Also having one out of context question
question_out_of_context = "IIM Calcutta's PGDBA program seems to be affiliated with Jio Institute, is that true?"

### Retrieval

#### Retrieval Based On Semantic Similarity Search

The vector store can be directly queried using `similarity_search` method. However, we will turn the vector store to a retriever using `as_retriever` method, and then use it. Benefit is that the resulting retriever is a *Runnable* object, making it easier to use with chains!

In [94]:
#Create retriever from vector store
retriever = vector_store_faiss.as_retriever()

In [95]:
type(retriever)

In [96]:
#Check default settings
docs = retriever.invoke(question)
docs

In [78]:
question

**Now let's pass a search parameter, that retrieves based on similarity score threshold.**

In [79]:
#Create retriever from vector store with similarity score threshold
retriever = vector_store_faiss.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.3}  # hyperparameter that can be tuned
)

In [80]:
docs = retriever.invoke(question)
docs

In [85]:
print(docs[0].page_content)

**Now let's pass another search parameter, that retrieves top k results only.**

In [90]:
#Create retriever from vector store with similarity score threshold
retriever = vector_store_faiss.as_retriever(
    search_kwargs={"k": 3} # hyperparameter that can be tuned
)

In [91]:
docs = retriever.invoke(question)
docs

**Now let's do metadata filtering.** 

A lot of vector stores also support filtering on metadata. Check more information on self-querying retrieval [here](https://python.langchain.com/docs/how_to/self_query/).

Thus, we can do both 1) **semantic search** by querying unstructured data directly 2) **metadata search** by applying structured query to metadata.

In [274]:
#Create retriever from vector store with metadata filter
retriever = vector_store_faiss.as_retriever(
    search_kwargs={"filter": {"source": "https://www.jioinstitute.edu.in/about/"}}
)

In [93]:
# all_splits_semantic

In [275]:
docs = retriever.invoke(question)
docs

We can see that the result from the other webpage **has been filtered out**!

#### Advanced Retrieval Based On Maximum Marginal Relevance

*Maximum marginal relevance* is a **re-ranking** algorithm that optimizes for both similarity to query *AND* diversity among selected documents. This can be used, if supported by the particular vector store.

In [98]:
#Original retrieval
for doc in docs:
    print(doc.page_content)
    print("-"*100)


In [99]:
retriever = vector_store_faiss.as_retriever(search_type="mmr")

In [100]:
docs = retriever.invoke(question)
docs

In [101]:
#Retrieval with MMR
for doc in docs:
    print(doc.page_content)
    print("-"*100)


#### Advanced Retrieval Based On Hybrid Search

*Hybrid search* combines the benefits of both standard semantic similarity search offered by LangChain, as well as other search techniques (such as BM25, full-text and so on). We'll do **BM25+Semantic Search** in our example below and we'll make use of ChromaDB for a change.

In [102]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

In [103]:
#Initialize BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents=all_splits_semantic)
bm25_retriever.k = 3 # Set the number of top documents to retrieve

In [104]:
#Initialize ChromaDB as a retriever
chroma_retriever = vector_store_chroma.as_retriever(search_kwargs={"k": 3})

In [105]:
#Perform reciprocal rank fusion (RRF)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.7, 0.3]
)

In [106]:
#Perform hybrid search
docs = ensemble_retriever.invoke(question)
docs

### Generation

**We'll use the advanced retrieval based on hybrid search, for our generation.**

In [107]:
#For reference
question

In [110]:
#For reference
question_out_of_context

#### Select the Chat Model

Conceptual Reference - [Chat models](https://python.langchain.com/docs/concepts/chat_models/)

In [111]:
from langchain.chat_models import init_chat_model

model_name = "gemini-flash-latest"
llm = init_chat_model(model_name, model_provider="google_genai") #Gemini replaces Groq/Llama here (other options: gemini-flash-lite-latest, gemini-pro-latest)

**Question - Can we choose an embedding model that has a different dimension than the chat (generation) model used for RAG?**

#### Create custom Chat Prompt Template

In [112]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """You are given the following information for reference:
{context}

Answer the following question:
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [113]:
chain = prompt | llm

In [354]:
response = chain.invoke({"context":docs,"question":question})
display(Markdown(response.text))

In [365]:
response = chain.invoke({"context":docs,"question":question_out_of_context})
display(Markdown(response.text))

#### Pull Chat Prompt Template from the Hub

In [114]:
from langchain_core.prompts import ChatPromptTemplate

# Equivalent of hub.pull("rlm/rag-prompt") - inlined because `langchain.hub` was removed
prompt_hub_rag = ChatPromptTemplate.from_messages([("human", "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")])
prompt_hub_rag

**Notice the prompt! This is how a good prompt should look like.** It explicity states to say *I don't know* if context is missing, which is a best practice.

In [367]:
chain = prompt_hub_rag | llm

In [115]:
response = chain.invoke({"context":docs,"question":question})
display(Markdown(response.text))

In [116]:
response = chain.invoke({"context":docs,"question":question_out_of_context})
display(Markdown(response.text))

#### Pull Chat Prompt Template from the Hub (and use LCEL!) - Recommended

Reference - [RAG chains](https://python.langchain.com/docs/how_to/sequence/)

In [117]:
# USE LCEL! (Recommended)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": ensemble_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke(question)
print(response)

In [118]:
#What is the type?
type(rag_chain)